# 05 - Backtesting Framework

This notebook demonstrates the backtesting framework for evaluating CNN model performance on historical data.

## Features Covered

1. **Standard Chronological Backtest** - Run predictions sequentially on historical data
2. **Walk-Forward Validation** - Test across multiple sequential periods
3. **Randomized Period Backtest** - Sample independent test periods for robust evaluation
4. **Performance Metrics** - Accuracy, Sharpe ratio, drawdown, win rate, and more
5. **Trade Simulation** - Realistic P&L with transaction costs
6. **Visualization** - Equity curves, drawdown charts, confusion matrices
7. **Report Generation** - Automated HTML reports

## Prerequisites

- A trained model saved in `models/best_model.pt`
- If no model is available, this notebook will use mock data for demonstration

In [ ]:
# Standard imports
import sys
import os
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
%matplotlib inline

print(f"Project root: {project_root}")

In [ ]:
# Import backtesting modules
from src.backtesting.engine import (
    BacktestEngine,
    BacktestConfig,
    BacktestResult,
    RandomizedBacktestConfig,
    RandomizedBacktestResult,
)
from src.backtesting.metrics import (
    calculate_prediction_metrics,
    calculate_trading_metrics,
    calculate_risk_metrics,
    calculate_monthly_returns,
)
from src.backtesting.simulator import TradeSimulator, SimulatorConfig
from src.backtesting.plots import (
    plot_equity_curve,
    plot_drawdown,
    plot_confusion_matrix,
    plot_returns_distribution,
    plot_monthly_returns,
    plot_signal_timeline,
    plot_win_rate_ci,
    create_backtest_dashboard,
)
from src.backtesting.report import generate_report, export_trades_csv, export_metrics_json

from src.utils.config import MODELS_DIR, PROJECT_ROOT

print("Backtesting modules loaded successfully!")

## 1. Check for Trained Model

In [ ]:
# Check if a trained model exists
model_path = MODELS_DIR / 'best_model.pt'
model_available = model_path.exists()

if model_available:
    print(f"Model found at: {model_path}")
    
    # Load predictor
    from src.prediction.predictor import Predictor
    predictor = Predictor(model_path)
    print("Predictor loaded successfully!")
else:
    print("No trained model found.")
    print("Train a model using notebook 02_training.ipynb first.")
    print("\nWe'll continue with demonstration using mock data.")
    predictor = None

## 2. Standard Chronological Backtest

Run predictions on historical data in chronological order, simulating real-world usage.

In [ ]:
# Configure backtest
config = BacktestConfig(
    # Date range (last year of data)
    start_date='2024-01-01',
    end_date='2024-12-31',
    
    # Capital and costs
    initial_capital=100000.0,
    commission=0.001,     # 0.1% per trade
    slippage=0.0005,      # 0.05% slippage
    
    # Strategy settings
    confidence_threshold=0.55,  # Only trade when confidence > 55%
    position_sizing='percentage',  # Use percentage of equity
    position_pct=0.1,     # 10% per trade
    allow_short=False,    # Long-only
    
    # Prediction settings
    horizon=5,            # T+5 predictions
    prediction_frequency=5,  # Predict every 5 days
    
    verbose=True,
)

print("Backtest Configuration:")
print(f"  Period: {config.start_date} to {config.end_date}")
print(f"  Initial Capital: ${config.initial_capital:,.0f}")
print(f"  Commission: {config.commission:.2%}")
print(f"  Confidence Threshold: {config.confidence_threshold:.0%}")
print(f"  Position Size: {config.position_pct:.0%} of equity")

In [ ]:
# Run backtest (if model is available)
if predictor is not None:
    # Initialize backtest engine
    engine = BacktestEngine(predictor=predictor)
    
    # Run backtest on AAPL
    ticker = 'AAPL'
    print(f"\nRunning backtest for {ticker}...")
    result = engine.run_backtest(ticker, config)
    
    print("\n" + "="*50)
    print(result.summary())
else:
    print("Skipping live backtest - no model available")
    result = None

## 3. Visualize Backtest Results

In [ ]:
if result is not None and result.equity_curve is not None and len(result.equity_curve) > 0:
    # Equity Curve
    fig = plot_equity_curve(result.equity_curve, title=f"{ticker} - Equity Curve")
    plt.show()
else:
    print("No equity curve data available for visualization")

In [ ]:
if result is not None and result.equity_curve is not None and len(result.equity_curve) > 0:
    # Drawdown Chart
    fig = plot_drawdown(result.equity_curve, title=f"{ticker} - Drawdown")
    plt.show()

In [ ]:
if result is not None and result.predictions:
    # Confusion Matrix
    preds = [p['predicted_direction'] for p in result.predictions if p.get('actual_direction', -1) >= 0]
    acts = [p['actual_direction'] for p in result.predictions if p.get('actual_direction', -1) >= 0]
    
    if preds and acts:
        fig = plot_confusion_matrix(preds, acts, title=f"{ticker} - Prediction Confusion Matrix")
        plt.show()

In [ ]:
if result is not None and result.returns is not None and len(result.returns) > 0:
    # Returns Distribution
    fig = plot_returns_distribution(result.returns, title=f"{ticker} - Returns Distribution")
    plt.show()

In [ ]:
if result is not None:
    # Create comprehensive dashboard
    fig = create_backtest_dashboard(result)
    plt.show()

## 4. Detailed Metrics Analysis

In [ ]:
if result is not None:
    print("="*60)
    print("PREDICTION METRICS")
    print("="*60)
    
    pred_metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'mcc']
    for metric in pred_metrics:
        if metric in result.metrics:
            print(f"{metric.replace('_', ' ').title():20s}: {result.metrics[metric]:.4f}")
    
    print(f"\nTrue Positives:  {result.metrics.get('true_positives', 0):5d}")
    print(f"True Negatives:  {result.metrics.get('true_negatives', 0):5d}")
    print(f"False Positives: {result.metrics.get('false_positives', 0):5d}")
    print(f"False Negatives: {result.metrics.get('false_negatives', 0):5d}")

In [ ]:
if result is not None:
    print("="*60)
    print("TRADING METRICS")
    print("="*60)
    
    trading_metrics = [
        ('Total Return', 'total_return', '{:.2%}'),
        ('Annualized Return', 'annualized_return', '{:.2%}'),
        ('Volatility (Annual)', 'volatility', '{:.2%}'),
        ('Sharpe Ratio', 'sharpe_ratio', '{:.2f}'),
        ('Sortino Ratio', 'sortino_ratio', '{:.2f}'),
        ('Calmar Ratio', 'calmar_ratio', '{:.2f}'),
        ('Max Drawdown', 'max_drawdown', '{:.2%}'),
        ('Win Rate', 'win_rate', '{:.2%}'),
        ('Profit Factor', 'profit_factor', '{:.2f}'),
        ('Average Win', 'avg_win', '{:.2%}'),
        ('Average Loss', 'avg_loss', '{:.2%}'),
    ]
    
    for label, key, fmt in trading_metrics:
        if key in result.metrics:
            print(f"{label:25s}: {fmt.format(result.metrics[key])}")

## 5. Walk-Forward Validation

Test model performance across multiple sequential time periods to evaluate stability.

In [ ]:
if predictor is not None:
    # Walk-forward configuration
    wf_config = BacktestConfig(
        start_date='2022-01-01',
        end_date='2024-12-31',
        initial_capital=100000.0,
        commission=0.001,
        confidence_threshold=0.55,
        position_sizing='percentage',
        position_pct=0.1,
        horizon=5,
        prediction_frequency=5,
        
        # Walk-forward settings
        walk_forward=True,
        train_periods=504,   # ~2 years
        test_periods=126,    # ~6 months
        
        verbose=True,
    )
    
    print("Running walk-forward validation...")
    wf_results = engine.run_walk_forward_backtest('AAPL', wf_config)
    
    print(f"\nCompleted {len(wf_results)} test periods")
else:
    print("Skipping walk-forward validation - no model available")
    wf_results = None

In [ ]:
if wf_results:
    # Summarize walk-forward results
    print("\nWalk-Forward Validation Summary")
    print("="*60)
    
    periods_data = []
    for i, res in enumerate(wf_results, 1):
        periods_data.append({
            'Period': i,
            'Start': res.start_date,
            'End': res.end_date,
            'Accuracy': res.metrics.get('accuracy', 0),
            'Return': res.metrics.get('total_return', 0),
            'Sharpe': res.metrics.get('sharpe_ratio', 0),
            'Max DD': res.metrics.get('max_drawdown', 0),
        })
    
    wf_df = pd.DataFrame(periods_data)
    
    # Format for display
    display_df = wf_df.copy()
    display_df['Accuracy'] = display_df['Accuracy'].apply(lambda x: f"{x:.1%}")
    display_df['Return'] = display_df['Return'].apply(lambda x: f"{x:.1%}")
    display_df['Sharpe'] = display_df['Sharpe'].apply(lambda x: f"{x:.2f}")
    display_df['Max DD'] = display_df['Max DD'].apply(lambda x: f"{x:.1%}")
    
    print(display_df.to_string(index=False))
    
    # Average metrics
    print("\n" + "-"*60)
    print(f"Average Accuracy: {wf_df['Accuracy'].mean():.1%}")
    print(f"Average Return:   {wf_df['Return'].mean():.1%}")
    print(f"Average Sharpe:   {wf_df['Sharpe'].mean():.2f}")
    print(f"Average Max DD:   {wf_df['Max DD'].mean():.1%}")

## 6. Randomized Period Backtest

Sample non-overlapping test periods randomly for robust performance evaluation that's less biased by specific market regimes.

In [ ]:
if predictor is not None:
    # Fetch data for randomized backtest
    from src.data.fetcher import fetch_stock_data
    
    ticker = 'AAPL'
    print(f"Fetching historical data for {ticker}...")
    
    # Get several years of data
    data = fetch_stock_data(ticker, '2018-01-01', '2024-12-31')
    print(f"Fetched {len(data)} rows")
    
    # Configure randomized backtest
    rand_config = RandomizedBacktestConfig(
        n_periods=100,         # Sample 100 test periods
        min_periods=50,        # Need at least 50 for valid test
        train_end_idx=int(len(data) * 0.7),  # First 70% for "training"
        
        random_seed=42,        # For reproducibility
        confidence_level=0.95, # 95% confidence intervals
        
        # Benchmark comparison
        benchmark_ticker='SPY',
        
        # Strategy settings
        commission=0.001,
        slippage=0.0005,
        confidence_threshold=0.5,
        horizon=5,
        
        verbose=True,
    )
    
    print(f"\nRunning randomized backtest with {rand_config.n_periods} periods...")
    rand_result = engine.run_randomized_backtest(ticker, data, rand_config)
    
    print("\n" + rand_result.summary())
else:
    print("Skipping randomized backtest - no model available")
    rand_result = None

In [ ]:
if rand_result is not None:
    # Visualize win rate with confidence interval
    fig = plot_win_rate_ci(
        rand_result.win_rate,
        rand_result.win_rate_ci[0],
        rand_result.win_rate_ci[1],
        title="Win Rate with 95% Confidence Interval"
    )
    plt.show()

In [ ]:
if rand_result is not None and rand_result.return_distribution:
    # Returns distribution
    from src.backtesting.plots import plot_period_returns_histogram
    
    benchmark_returns = [p.benchmark_return for p in rand_result.periods 
                         if p.benchmark_return is not None]
    
    fig = plot_period_returns_histogram(
        rand_result.return_distribution,
        benchmark_returns if benchmark_returns else None,
        title="Model vs Benchmark Returns Distribution"
    )
    plt.show()

## 7. Generate Report

In [ ]:
if result is not None:
    # Generate HTML report
    reports_dir = PROJECT_ROOT / 'reports'
    reports_dir.mkdir(exist_ok=True)
    
    report_path = reports_dir / f'backtest_{ticker}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.html'
    
    print("Generating HTML report...")
    generate_report(result, report_path, format='html')
    print(f"Report saved to: {report_path}")
    
    # Export trades to CSV
    if result.trades:
        trades_path = reports_dir / f'trades_{ticker}_{datetime.now().strftime("%Y%m%d")}.csv'
        export_trades_csv(result.trades, trades_path)
        print(f"Trade log saved to: {trades_path}")
    
    # Export metrics to JSON
    metrics_path = reports_dir / f'metrics_{ticker}_{datetime.now().strftime("%Y%m%d")}.json'
    export_metrics_json(result.metrics, metrics_path)
    print(f"Metrics saved to: {metrics_path}")

## 8. Multi-Stock Backtest Comparison

Compare model performance across multiple stocks.

In [ ]:
if predictor is not None:
    # Test multiple stocks
    tickers_to_test = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META']
    
    comparison_results = []
    
    for ticker in tickers_to_test:
        print(f"\nBacktesting {ticker}...")
        try:
            res = engine.run_backtest(ticker, BacktestConfig(
                start_date='2024-01-01',
                end_date='2024-12-31',
                initial_capital=100000.0,
                confidence_threshold=0.55,
                position_sizing='percentage',
                position_pct=0.1,
                horizon=5,
                prediction_frequency=5,
                verbose=False,
            ))
            
            comparison_results.append({
                'Ticker': ticker,
                'Accuracy': res.metrics.get('accuracy', 0),
                'Return': res.metrics.get('total_return', 0),
                'Sharpe': res.metrics.get('sharpe_ratio', 0),
                'Max DD': res.metrics.get('max_drawdown', 0),
                'Win Rate': res.metrics.get('win_rate', 0),
                'Trades': len(res.trades),
            })
            print(f"  Accuracy: {res.metrics.get('accuracy', 0):.1%}, Return: {res.metrics.get('total_return', 0):.1%}")
        except Exception as e:
            print(f"  Error: {e}")
    
    if comparison_results:
        comparison_df = pd.DataFrame(comparison_results)
        print("\n" + "="*60)
        print("MULTI-STOCK COMPARISON")
        print("="*60)
        
        # Format for display
        display_df = comparison_df.copy()
        display_df['Accuracy'] = display_df['Accuracy'].apply(lambda x: f"{x:.1%}")
        display_df['Return'] = display_df['Return'].apply(lambda x: f"{x:.1%}")
        display_df['Sharpe'] = display_df['Sharpe'].apply(lambda x: f"{x:.2f}")
        display_df['Max DD'] = display_df['Max DD'].apply(lambda x: f"{x:.1%}")
        display_df['Win Rate'] = display_df['Win Rate'].apply(lambda x: f"{x:.1%}")
        
        print(display_df.to_string(index=False))
else:
    print("Skipping multi-stock comparison - no model available")

## 9. Interpreting Results

### Key Metrics to Consider

1. **Accuracy**: Percentage of correct predictions. >50% is better than random.
2. **Sharpe Ratio**: Risk-adjusted return. >1.0 is generally good.
3. **Max Drawdown**: Largest peak-to-trough decline. Lower is better.
4. **Win Rate**: Percentage of profitable trades.
5. **Profit Factor**: Gross profit / gross loss. >1.0 is profitable.

### Important Considerations

- **Transaction Costs**: Real trading involves commissions and slippage.
- **Market Regime**: Performance varies in different market conditions.
- **Overfitting**: Good backtest results don't guarantee future performance.
- **Sample Size**: More trades = more reliable statistics.

### Disclaimer

This backtesting framework is for educational purposes only. Past performance does not guarantee future results. Always do your own research before making investment decisions.

In [ ]:
print("\nBacktesting notebook complete!")
print("\nNext steps:")
print("1. Review the generated reports in the 'reports/' directory")
print("2. Try different configuration parameters")
print("3. Test on different stocks and time periods")
print("4. Use the Streamlit dashboard for interactive backtesting")